In [ ]:
import torch
import torch.nn.functional as F

#ChatGPT generated

def get_synthetic_batch(batch_size=4, seq_len=32, vocab_size=1000, seed=0):
    torch.manual_seed(seed)
    return {
        "input_ids": torch.randint(0, vocab_size, (batch_size, seq_len)),
        "attention_mask": torch.ones(batch_size, seq_len)
    }


def extract_outputs(model, batch):
    with torch.no_grad():
        outputs = model(
            **batch,
            output_attentions=True,   # IMPORTANT
            return_dict=True
        )

    hidden = outputs.last_hidden_state

    logits = getattr(outputs, "logits", None)
    attentions = getattr(outputs, "attentions", None)

    return hidden, logits, attentions


def assert_models_equivalent(
    model_a,
    model_b,
    atol_hidden=1e-5,
    atol_logits=1e-4,
    cosine_threshold=0.999,
    attention_cosine_threshold = 0.999,
    check_attention=False
):
    """
    Compares two models for functional equivalence.
    """

    model_a.eval()
    model_b.eval()

    batch = get_synthetic_batch()

    h_a, l_a, attn_a = extract_outputs(model_a, batch)
    h_b, l_b, attn_b = extract_outputs(model_b, batch)

    results = {}

    passed = True

    # -------------------------
    # 1. Hidden state check
    # -------------------------
    if h_a is not None and h_b is not None:
        hidden_mse = F.mse_loss(h_a, h_b).item()
        results["hidden_mse"] = hidden_mse

        if hidden_mse > atol_hidden:
            print(f"Hidden states differ too much: {hidden_mse:.6g}, with lower threshold: {atol_hidden}")
            passed = False

    # -------------------------
    # 2. Logit similarity
    # -------------------------
    if l_a is not None and l_b is not None:
        cos = F.cosine_similarity(
            l_a.flatten(),
            l_b.flatten(),
            dim=0
        ).item()

        results["logit_cosine"] = cos

        if cos < cosine_threshold:
            print(f"Logits not similar enough: {cos:.6g}, with upper threshold: {cosine_threshold}")
            passed = False
        
        logit_mse = F.mse_loss(l_a, l_b).item()
        results["logit_mse"] = logit_mse

        if logit_mse > atol_logits:
            print(f"Logits differ too much: {logit_mse:.6g}, with lower threshold: {atol_logits}")

    # -------------------------
    # 3. Attention check
    # -------------------------

    #TODO: adjust thi
    if check_attention and attn_a is not None and attn_b is not None:
        attn_diffs = []

        for a, b in zip(attn_a, attn_b):
            attn_diffs.append((a - b).abs().mean().item())

        attn_diff = sum(attn_diffs) / len(attn_diffs)
        results["raw_attention_diff"] = attn_diff

        a = torch.cat([x.flatten() for x in attn_a])
        b = torch.cat([x.flatten() for x in attn_b])

        attn_cos = F.cosine_similarity(a, b, dim=0).item()
        results["attention_cosine"] = attn_cos

        if attn_cos < attention_cosine_threshold:
            print(f"Attention not similar enough: {attn_cos}, with lower threshold: {attention_cosine_threshold}")
            passed = False

    if passed:
        print("Models are equivalent within tolerance")
    else:
        print("Models are NOT equivalent within tolerance")
        
    for k, v in results.items():
        print(f"{k}: {v:.6g}")

    return results

In [6]:
import huggingface_hub as hb
from transformers import BertModel

model = BertModel.from_pretrained("prajjwal1/bert-tiny", attn_implementation="eager")

Loading weights: 100%|██████████| 39/39 [00:00<00:00, 19459.65it/s]
[transformers] BertModel LOAD REPORT from: prajjwal1/bert-tiny
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
from module_disentanglement import (disentangle_module, print_layers_with_params, test_disentanglement_form)


disentangled_model = disentangle_module(model, 0.1)
print_layers_with_params(model)
print()
print_layers_with_params(disentangled_model)

query: Linear(in_features=128, out_features=128, bias=True)
[W1]step=0000 cost=5.743953e+01 entropy=4.341038 effective_rank=76.7872
[W1]step=0010 cost=1.992956e+01 entropy=3.488519 effective_rank=32.7374
[W1]step=0020 cost=1.292824e+01 entropy=3.225918 effective_rank=25.1767
[W1]step=0030 cost=1.003055e+01 entropy=3.102000 effective_rank=22.2424
[W1]step=0040 cost=8.569083e+00 entropy=3.036208 effective_rank=20.8261
[W1]step=0050 cost=7.735552e+00 entropy=2.999362 effective_rank=20.0727
[W1]step=0060 cost=7.238331e+00 entropy=2.977284 effective_rank=19.6344
[W1]step=0070 cost=6.902623e+00 entropy=2.960930 effective_rank=19.3159
[W1]step=0080 cost=6.687886e+00 entropy=2.950817 effective_rank=19.1216
[W1]step=0090 cost=6.531128e+00 entropy=2.941966 effective_rank=18.9531
[W1]step=0100 cost=6.409007e+00 entropy=2.935232 effective_rank=18.8259
key: Linear(in_features=128, out_features=128, bias=True)
[W1]step=0000 cost=5.634274e+01 entropy=4.355157 effective_rank=77.8790
[W1]step=0010 cost

In [8]:
results = assert_models_equivalent(model, disentangled_model, check_attention=True)

Hidden states differ too much: 0.5083392858505249, with lower threshold: 1e-05
Models are NOT equivalent within tolerance
hidden_mse: 0.508339
attention_diff: 0.0136287
